## CoAtNet

### Review:

* Brings in the best of both worlds. CNNS are good at local geatures and have nice inductive biases. Transformers are good at global relationships. If you stack CNNs and transformer blocks, you get good results. 
* **Relative Attention**. Take away the absolute positional embeddings like we had in the vanilla ViT, and instead use relative position biases in attentnions scores. This generalizes nicely to different input sizes. 

In [2]:
from torchvision.datasets import CIFAR100
import torchvision.transforms as transforms
import torch
import torch.nn as nn
import torch.nn.functional as F # This gives us the softmax()
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((56, 56)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

train_dataset = CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR100(root='./data', train=False, download=True, transform=transform)

In [3]:
from vit import ViT, PatchEmbedding, MultiHeadAttention, TransformerEncoder ,ClassificationHead , FeedForward                                                               

We basically need to replace the PatchEmbedding with a deeper CNN

In [ ]:
class CNNStem(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.Conv2d(128, num_hiddens, kernel_size =3, stride = 2, padding =1),
            nn.BatchNorm2d(num_hiddens),
            nn.GELU()
        )
    def forward(self, x):
        x = self.stem(x)
        return x
     

In [25]:
img = torch.randn(1, 3, 56, 56)

In [8]:
stem = CNNStem(num_hiddens = 512)
outs = stem(x)
print(outs.shape)

torch.Size([2, 512, 7, 7])


In [40]:
class Vit(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_classes, depth=6, dropout=0.1):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embedding = PatchEmbedding(img_size, patch_size, num_hiddens)
        self.cls_token = nn.Parameter(torch.randn(1, 1, num_hiddens))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))
        self.encoder = TransformerEncoder(num_hiddens, depth, num_heads, mlp_dim=num_hiddens * 4)
        self.classification_head = ClassificationHead(num_hiddens, num_classes)

    def forward(self, x):
        batch = x.shape[0]
        patches = self.patch_embedding(x)
        print(patches.shape)
        patches = patches.flatten(2).transpose(1, 2)
        print(patches.shape)
        cls_tokens = self.cls_token.expand(batch, -1, -1)
        x = torch.cat((cls_tokens, patches), dim=1)
        x = x + self.pos_embedding
        x = self.encoder(x)
        x = self.classification_head(x)
        return x
    
class CoAtNEt(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_classes, depth=6, dropout=0.1):
        super().__init__()
        self.num_patches = 7*7
        self.stem = CNNStem(num_hiddens)
        self.cls_token = nn.Parameter(torch.randn(1, 1, num_hiddens))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, num_hiddens))
        self.encoder = TransformerEncoder(num_hiddens, depth, num_heads, mlp_dim=num_hiddens * 4)
        self.classification_head = ClassificationHead(num_hiddens, num_classes)
        self.patch_size = patch_size

    def forward(self, x):
        batch = x.shape[0]
        patches_equivalent = self.stem(x)
        patches_equivalent = patches_equivalent.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(batch, -1, -1)
        x = torch.cat((cls_tokens, patches_equivalent), dim=1)
        x = x + self.pos_embedding
        x = self.encoder(x)
        x = self.classification_head(x)
        return x

In [41]:
net = Vit(patch_size = 7, num_hiddens=512, num_heads = 8, num_classes= 100, img_size = 56)
coatnet = CoAtNEt(patch_size = 7, num_hiddens=512, num_heads = 8, num_classes= 100, img_size = 56)

In [38]:
x = net(img)

torch.Size([1, 512, 8, 8])
torch.Size([1, 64, 512])


In [39]:
x = coatnet(img)

torch.Size([1, 512, 7, 7])
torch.Size([1, 49, 512])


In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using: {device}")

Using: mps


In [43]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import time
optimizer = torch.optim.Adam(coatnet.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    coatnet.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = coatnet(images)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)
    elapsed = time.time() - start
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f} | Time: {elapsed:.1f}s")